# Рамки и маски: числовая практика

Дополнение к § 3.2. На небольших координатах проверяем порог, площадь и перекрытие. Для вычислений нужен NumPy. Готовую модель сначала запустите на `data/coffee.png`, затем проверьте её на собственных фотографиях.

## Два режима работы

Числовые упражнения работают без скачивания весов. Для запуска модели установите пакеты из `requirements.txt` со страницы главы и распакуйте `images.zip` рядом с ноутбуком: изображения окажутся в папке `data/`. В Colab выполните `%pip install -r requirements.txt`. Затем смените `run_model = False` на `True` и запустите ячейки сверху вниз. Первый запуск скачивает веса; на CPU он может занять несколько минут. После тестового кадра замените путь на собственное фото.

## Отбор рамок
Поменяйте порог и проследите, какие индексы исчезают.

In [ ]:
import numpy as np
boxes = np.array([[40,30,180,160],[190,40,226,150],[42,31,178,159]])
scores = np.array([0.84, 0.31, 0.73])
for threshold in (0.30, 0.50, 0.80):
    print(threshold, np.flatnonzero(scores >= threshold).tolist())
assert (boxes[:,2] > boxes[:,0]).all()

## Перекрытие
Сравните первую и третью рамки. При одинаковых рамках функция должна вернуть 1, при далёких — 0.

In [ ]:
def iou(a, b):
    left = max(a[0], b[0]); top = max(a[1], b[1])
    right = min(a[2], b[2]); bottom = min(a[3], b[3])
    inter = max(0, right-left) * max(0, bottom-top)
    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])
    return inter / (area_a + area_b - inter)
print(round(iou(boxes[0], boxes[2]), 3))
assert iou(boxes[0], boxes[0]) == 1
assert iou(boxes[0], [1000,1000,1010,1010]) == 0

## Пиксели маски
Почему число истинных пикселей меняется с порогом?

In [ ]:
mask_scores = np.array([[0.1,0.7,0.8],[0.2,0.6,0.3]])
for threshold in (0.5,0.75):
    print(threshold, int((mask_scores >= threshold).sum()))

## Опыт с фотографией из набора

Сначала запустите детектор и сегментатор на `data/coffee.png`. Сопоставьте рамки и маску с исходной чашкой, затем сравните число рамок при порогах 0,30 и 0,70. В статье показано, как наложить результаты на исходный кадр. После этого выберите собственную фотографию с двумя предметами, метки которых могут быть в COCO. Перед запуском запишите, что и где ожидаете увидеть, и замените имя файла в коде.

In [ ]:
run_model = False
if run_model:
    from PIL import Image
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    import torch
    from torchvision.transforms.functional import to_tensor
    from torchvision.models.detection import (
        FasterRCNN_ResNet50_FPN_V2_Weights, fasterrcnn_resnet50_fpn_v2,
        MaskRCNN_ResNet50_FPN_V2_Weights, maskrcnn_resnet50_fpn_v2,
    )
    image = Image.open("data/coffee.png").convert("RGB")
    tensor = to_tensor(image)
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    detector = fasterrcnn_resnet50_fpn_v2(weights=weights).eval()
    with torch.inference_mode():
        result = detector([tensor])[0]
    names = weights.meta["categories"]
    report = pd.DataFrame([
        {"class": names[int(label)], "score": float(score),
         "box": [round(x) for x in box.tolist()]}
        for box, label, score in zip(result["boxes"], result["labels"], result["scores"])
    ], columns=["class", "score", "box"])
    print(report.head(10).to_string())
    fig, axes = plt.subplots(2, 1, figsize=(7, 9))
    for ax, threshold in zip(axes, (0.30, 0.70)):
        ax.imshow(image)
        for index, (box, label, score) in enumerate(zip(
            result["boxes"], result["labels"], result["scores"])):
            if float(score) < threshold: continue
            x1, y1, x2, y2 = box.tolist()
            ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                                   fill=False, edgecolor="tab:blue", linewidth=2))
            ax.text(x1 + 3, y1 + 16, str(index), fontsize=13,
                    bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"})
        ax.set_title(f"Порог {threshold:.2f}"); ax.axis("off")
    plt.show()
    mask_weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    segmenter = maskrcnn_resnet50_fpn_v2(weights=mask_weights).eval()
    with torch.inference_mode():
        segmented = segmenter([tensor])[0]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.imshow(image)
    selected = torch.where(segmented["scores"] >= 0.50)[0]
    if len(selected):
        i = int(selected[0])
        binary_mask = (segmented["masks"][i, 0] >= 0.50).numpy()
        assert binary_mask.shape == (image.height, image.width)
        ax.imshow(np.ma.masked_where(~binary_mask, binary_mask),
                  cmap="Blues", vmin=0, vmax=1, alpha=0.5)
        label = mask_weights.meta["categories"][int(segmented["labels"][i])]
        ax.set_title(f"Маска: {label}")
        print("Класс маски:", label, "пикселей:", int(binary_mask.sum()))
    else:
        ax.set_title("Нет объектов выше порога 0,50")
    ax.axis("off"); plt.show()


## Сравнение порогов

Для тестового и собственного снимка запишите число рамок при порогах 0,30 и 0,70, затем сверьте каждое предсказание с исходным фото. Отдельно отметьте пропущенные предметы и лишние рамки. У Mask R-CNN сравните маску хотя бы одного найденного объекта с его рамкой и проверьте, совпадает ли размер маски с размером изображения.

In [ ]:
if run_model:
    for threshold in (0.30, 0.70):
        print("Порог:", threshold)
        print(report[report["score"] >= threshold].to_string(index=False))


In [ ]:
if run_model:
    from importlib.metadata import version
    print("Библиотеки:", {name: version(name) for name in
          ["torch", "torchvision", "transformers", "numpy", "pandas"]})
    print("Веса:", weights.url, mask_weights.url)


## Самопроверка




### 1. Какую информацию содержит предсказание детектора для одного объекта?

A. Только метку класса.

B. Только четыре координаты.

C. Класс, оценку и рамку.

D. Маску каждого пикселя всего изображения.




### 2. Рамка задана координатами $(40,30,180,160)$. Каковы её ширина и площадь?

A. 140 пикселей и 18 200 квадратных пикселей.

B. 180 пикселей и 25 200 квадратных пикселей.

C. 140 пикселей и 18 060 квадратных пикселей.

D. 130 пикселей и 18 200 квадратных пикселей.




### 3. Для рамок $A=(0,0,4,3)$ и $B=(2,1,6,4)$ площадь пересечения равна 4, а площадь объединения — 20. Чему равен IoU?

A. 0,8.

B. 0,5.

C. 0,25.

D. 0,2.




### 4. Оценки трёх рамок равны 0,84, 0,42 и 0,31. Сколько рамок будет показано при пороге 0,50?

A. Три.

B. Одна.

C. Две.

D. Ни одной.




### 5. В выводе нет рамок при пороге оценки 0,70. Как корректно интерпретировать результат?

A. На фотографии точно нет объектов.

B. Модель не загрузилась.

C. Нет предсказаний с оценкой не ниже 0,70.

D. Все маски состоят из нулей.




### 6. При каком значении IoU рамки не имеют общей площади?

A. 0.

B. 0,5.

C. 1.

D. 2.




### 7. Что лучше описывает маска объекта по сравнению с рамкой?

A. Маска задаёт только класс предмета.

B. Маска всегда покрывает весь кадр.

C. Маска заменяет оценку детектора.

D. Маска отмечает пиксели, отнесённые к объекту.




### 8. Порог показа равен 0,30. Предсказания имеют оценки 0,31, 0,42 и 0,84. Что произойдёт, если поднять порог до 0,70?

A. Добавятся рамки с оценками 0,31 и 0,42.

B. Останется одна рамка с оценкой 0,84.

C. Все три рамки останутся видимыми.

D. Изменятся координаты всех рамок.




### 9. В примере маска выбранного объекта имеет форму $1\times H\times W$. Что обозначают эти размерности?

A. Один кадр, одна рамка и одна оценка.

B. Число классов, ширина и высота.

C. Один объект, высота и ширина изображения.

D. Один объект, число каналов цвета и число классов.




### 10. На одном снимке модель выдала ложную рамку. Какое действие даст более содержательную оценку качества?

A. Проверить предсказания на нескольких размеченных снимках и отдельно учесть пропуски и ложные рамки.

B. Сразу признать модель непригодной для любых задач.

C. Удалить снимок из проверки и оставить только удачные.

D. Повысить порог до тех пор, пока ложная рамка не исчезнет, и считать проверку завершённой.




## Подсказки и решения

### Вопрос 1

**Подсказка**


Сопоставьте предсказание с тремя столбцами учебной таблицы.



**Решение**


**C.** Детектор связывает класс и оценку с координатами рамки.



### Вопрос 2

**Подсказка**


Вычислите $x_2-x_1$ и $y_2-y_1$, затем перемножьте размеры.



**Решение**


**A.** Ширина равна $180-40=140$, высота — $160-30=130$, площадь — $140\cdot130=18\,200$ квадратных пикселей.



### Вопрос 3

**Подсказка**


Разделите площадь пересечения на площадь объединения.



**Решение**


**D.** $operatorname{IoU}=4/20=0{,}2$.



### Вопрос 4

**Подсказка**


Оставьте оценки, которые не меньше порога.



**Решение**


**B.** Порог проходит только оценка 0,84.



### Вопрос 5

**Подсказка**


Порог фильтрует предсказания модели, а не сами объекты на фотографии.



**Решение**


**C.** Пустой вывод означает, что ни одна оценка не прошла фильтр; он не подтверждает отсутствие предметов.



### Вопрос 6

**Подсказка**


Если площадь пересечения равна нулю, чему равно отношение пересечения к объединению?



**Решение**


**A.** Без пересечения числитель равен нулю, поэтому IoU равен 0.



### Вопрос 7

**Подсказка**


Рамка охватывает объект прямоугольником; маска уточняет его форму.



**Решение**


**D.** Маска содержит оценку принадлежности для пикселей и позволяет описать контур объекта.



### Вопрос 8

**Подсказка**


Сравните каждую оценку с новым порогом.



**Решение**


**B.** Только 0,84 не меньше 0,70; остальные кандидаты скрываются.



### Вопрос 9

**Подсказка**


Первое измерение соответствует одному экземпляру объекта, оставшиеся — сетке пикселей.



**Решение**


**C.** Для одного объекта маска хранит значения на сетке размером $H\times W$.



### Вопрос 10

**Подсказка**


Один пример показывает конкретную ошибку, но не её частоту и не количество пропусков.



**Решение**


**A.** Набор размеченных снимков позволяет последовательно оценить и ложные обнаружения, и пропуски при зафиксированном пороге.



